# 🎓 Day 1 · Session 2
# Working with Modern LLMs
## Frontier Models, Open-Source Models & Universal APIs

In Session 1, we understood how LLMs work internally.  
In this session, we move from **understanding models** to **using models through code**.

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Understand cloud, local and hybrid LLM usage
- Call OpenAI models using Python
- Understand `system`, `user`, and `assistant` message roles
- Compare model behavior using the same prompt
- Use streaming responses
- Build a simple multi-turn teaching assistant
- Understand where Ollama/local models fit
- Apply a model selection framework

## 📦 Install Required Packages

Run only if packages are missing.

```bash
pip install openai python-dotenv pandas
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas

## 📚 Imports

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
from openai import OpenAI

## 🔑 Load API Key from `.env`

Expected `.env` inside `day1-llm-foundations`:

```env
OPENAI_API_KEY=sk-your-key-here
```

Optional later:

```env
DEEPSEEK_API_KEY=your-key
ANTHROPIC_API_KEY=your-key
GOOGLE_API_KEY=your-key
```

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)

openai_key = os.getenv("OPENAI_API_KEY")

if not openai_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

print("OPENAI_API_KEY loaded successfully.")
print("Key starts with:", openai_key[:10], "...")

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client initialized successfully.")

# 1️⃣ What is an API?

An API is a bridge between your application and another service.

```text
Your Python Code
      ↓
LLM API
      ↓
OpenAI / Claude / Gemini / DeepSeek
      ↓
Model Response
```

In simple terms:

> Your code sends a prompt. The model sends back a response.

# 2️⃣ Where Does the Model Run?

| Option | Where it runs | Examples | Best For |
|---|---|---|---|
| ☁️ Cloud | Provider servers | OpenAI, Claude, Gemini, DeepSeek API | Best quality, easiest start |
| 💻 Local | Your laptop/server | Ollama, LM Studio, llama.cpp | Privacy, offline use, zero API cost |
| 🏢 Hybrid | Mix of cloud and local | Enterprise architecture | Cost, compliance, flexibility |

Key idea:

> Choosing a model is not only about intelligence. It is also about **cost, privacy, speed and control**.

# 3️⃣ Your First OpenAI API Call

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain APIs in one simple paragraph for engineering faculty."}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

# 4️⃣ Message Roles: System, User and Assistant

| Role | Meaning |
|---|---|
| `system` | Instructions, persona and rules |
| `user` | Human question or task |
| `assistant` | Previous model response / conversation history |

Important:

> The assistant message is not a separate prompt. It represents what the AI said earlier.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor for engineering students. Use simple examples."
    },
    {
        "role": "user",
        "content": "Explain Newton's Second Law in simple terms."
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

print(response.choices[0].message.content)

## 🧪 Mini Demo: Why Assistant History Matters

In [ ]:
# Case 1: No conversation history
messages_without_history = [
    {"role": "user", "content": "Give another example."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_without_history,
    temperature=0.3
)

print(response.choices[0].message.content)

In [ ]:
# Case 2: With conversation history
messages_with_history = [
    {"role": "user", "content": "Explain Newton's Second Law in simple terms."},
    {"role": "assistant", "content": "Newton's Second Law says that force equals mass times acceleration."},
    {"role": "user", "content": "Give another example."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_with_history,
    temperature=0.3
)

print(response.choices[0].message.content)

## 💬 Discussion

Ask participants:

1. Why did the first answer lack context?
2. Why did the second answer understand "another example"?
3. What happens if a conversation becomes very long?
4. How does this connect to context window?

# 5️⃣ Reusable Helper Function

In [ ]:
def ask_openai(
    prompt,
    model="gpt-4o-mini",
    temperature=0.3,
    system_prompt="You are a helpful AI teaching assistant.",
    max_tokens=500
):
    """
    Sends a prompt to an OpenAI chat model and returns the response text.
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [ ]:
print(ask_openai("Explain the difference between AI, ML and Deep Learning in 5 bullet points."))

# 6️⃣ Same Prompt, Different Temperatures

In [ ]:
prompt = "Create one catchy title for a 5-day FDP on Agentic AI for AI and ML faculty."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print("=" * 80)
    print("Temperature:", temp)
    print("=" * 80)
    print(ask_openai(prompt, temperature=temp, max_tokens=100))
    print()

## 💬 Discussion

Which temperature would you choose for:

- Code generation?
- Creative workshop title?
- Policy summarization?
- Exam question generation?
- Student feedback?

Rule of thumb:

> Correctness needed → lower temperature  
> Creativity needed → higher temperature

# 7️⃣ Same Prompt, Different Model Choices

In [ ]:
comparison_prompt = '''
Explain Retrieval-Augmented Generation (RAG) to an AI and ML professor.
Use:
1. A one-line definition
2. A simple analogy
3. One real-world university use case
'''

models_to_test = ["gpt-4o-mini"]

# Uncomment if your account has access and you want to compare.
# models_to_test.append("gpt-4o")

for model in models_to_test:
    print("=" * 80)
    print("Model:", model)
    print("=" * 80)
    print(ask_openai(comparison_prompt, model=model, temperature=0.3, max_tokens=400))
    print()

## 🧪 AI Mythbusters #2

### Myth:
> The most expensive model is always the best model.

### Reality:
Not always.

For many teaching tasks:
- cheaper models are good enough
- faster models improve classroom experience
- local models may be better for privacy
- long-context models are useful only when needed

# 8️⃣ Streaming Output

In [ ]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain why streaming output improves chatbot user experience in 5 bullet points."}
    ],
    temperature=0.3,
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

# 9️⃣ Multi-Turn Teaching Assistant

In [ ]:
teaching_messages = [
    {
        "role": "system",
        "content": "You are a patient AI tutor for B.Tech students. Explain clearly, ask one follow-up question, and avoid unnecessary jargon."
    }
]

def chat_with_tutor(user_message):
    """
    Adds user message, calls the model, stores assistant reply, and returns it.
    """
    teaching_messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=teaching_messages,
        temperature=0.4,
        max_tokens=500
    )

    reply = response.choices[0].message.content
    teaching_messages.append({"role": "assistant", "content": reply})
    return reply

In [ ]:
print(chat_with_tutor("Explain overfitting using a student exam preparation analogy."))

In [ ]:
print(chat_with_tutor("Now give me one example from machine learning."))

In [ ]:
print(chat_with_tutor("Summarize our discussion so far in 3 bullet points."))

## 🔍 Observe

The model understood "our discussion so far" because we sent the full message history.

Trade-off:

> Longer conversation = more tokens = more cost = more context usage.

# 🔟 Local Models with Ollama

Ollama lets you run open-source models locally.

Before running this demo:

```bash
ollama pull llama3.2
ollama serve
```

In [ ]:
# This cell is optional.
# It works only if Ollama is installed and running locally.

RUN_OLLAMA = False

if RUN_OLLAMA:
    ollama_client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )

    response = ollama_client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "user", "content": "Explain local LLMs in one paragraph."}
        ],
        temperature=0.3
    )

    print(response.choices[0].message.content)
else:
    print("Skipping Ollama demo. Set RUN_OLLAMA = True after installing and starting Ollama.")

# 1️⃣1️⃣ OpenAI-Compatible Provider Pattern

In [ ]:
def create_openai_compatible_client(provider):
    """
    Creates clients for OpenAI-compatible providers.
    This function is for teaching purposes.
    """
    provider = provider.lower()

    if provider == "openai":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    elif provider == "ollama":
        return OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

    elif provider == "deepseek":
        deepseek_key = os.getenv("DEEPSEEK_API_KEY")
        if not deepseek_key:
            raise ValueError("DEEPSEEK_API_KEY not found in .env")
        return OpenAI(
            base_url="https://api.deepseek.com",
            api_key=deepseek_key
        )

    else:
        raise ValueError(f"Unknown provider: {provider}")

## Note on Claude and Gemini

Claude and Gemini have their own official SDKs.

Some gateways offer OpenAI-compatible access, but for teaching fundamentals, it is better to first learn:

- OpenAI style API
- Native SDKs where required
- Local model access through Ollama

# 1️⃣2️⃣ Cost Awareness

In [ ]:
cost_data = [
    {"Scenario": "Single student explanation", "Approx Input Tokens": 300, "Approx Output Tokens": 300, "Risk": "Low"},
    {"Scenario": "100 students asking 5 questions each", "Approx Input Tokens": 300 * 500, "Approx Output Tokens": 300 * 500, "Risk": "Medium"},
    {"Scenario": "Agent running tools in a loop", "Approx Input Tokens": "Can grow quickly", "Approx Output Tokens": "Can grow quickly", "Risk": "High"},
    {"Scenario": "RAG chatbot with large documents", "Approx Input Tokens": "High because context is included", "Approx Output Tokens": "Medium", "Risk": "Medium to High"},
]

pd.DataFrame(cost_data)

# 1️⃣3️⃣ Model Selection Framework

In [ ]:
model_selection = [
    {"Use Case": "Quick teaching explanation", "Recommended Option": "GPT-4o-mini / Gemini Flash", "Why": "Low cost, fast, good quality"},
    {"Use Case": "Sensitive student data", "Recommended Option": "Local model using Ollama", "Why": "Data stays inside institution"},
    {"Use Case": "Long research paper analysis", "Recommended Option": "Claude / Gemini long-context model", "Why": "Large context window"},
    {"Use Case": "Complex coding support", "Recommended Option": "GPT-4o / Claude Sonnet", "Why": "Better reasoning and code quality"},
    {"Use Case": "Budget-constrained experimentation", "Recommended Option": "GPT-4o-mini / DeepSeek / Ollama", "Why": "Cost-effective"},
]

pd.DataFrame(model_selection)

## 💬 Faculty Discussion

Your college wants to build four AI systems:

1. Student FAQ chatbot  
2. Research paper summarizer  
3. Question paper generator  
4. Private HR policy assistant  

Which model option would you choose for each?

Expected discussion:
- FAQ chatbot → cheap cloud or local + RAG
- Research summarizer → long-context model
- Question paper generator → cheap frontier model, with faculty review
- HR policy assistant → local or secure enterprise setup

# 1️⃣4️⃣ AI Pulse: Why Open-Source Models Matter

Discussion topic:

> Why would companies release powerful models for free?

Possible reasons:

- Build developer ecosystem
- Compete with closed providers
- Drive hardware/cloud adoption
- Encourage research adoption
- Create standards and mindshare

Key message:

> Open-source models make AI education more accessible.

# 1️⃣5️⃣ Lab Assignment

## Build Your Multi-Provider AI Assistant

### Task 1
Call `gpt-4o-mini` with a question from your subject area.

### Task 2
Change the system prompt so the model behaves like:
- a strict examiner
- a friendly tutor
- a research guide

Compare outputs.

### Task 3
Build a 3-turn conversation using the `assistant` role as conversation history.

### Task 4
Try streaming output.

### Task 5
Optional: run the same prompt using Ollama.

### Bonus
Create a simple model selection table for your department's AI use cases.

# ✅ Key Takeaways

1. Most AI apps talk to models through APIs.
2. Cloud models are easy and powerful, but cost and privacy matter.
3. Local models are useful for privacy, offline labs and low-cost experimentation.
4. The OpenAI-style message format is widely used.
5. `system` defines behavior, `user` gives the task, and `assistant` stores previous model responses.
6. Streaming improves user experience.
7. Model choice depends on task, cost, quality, privacy and context window.
8. The most expensive model is not always the best model.